# Chapter 3 · Abstraction Boundary Lab

Predict before every run. The goal is to distinguish implementation replacement from contract violation.

In [ ]:
from collections.abc import Mapping

def gross_margin_from_mapping(source: Mapping[str, float]) -> float:
    revenue = source['revenue']
    gross_profit = source['gross_profit']
    if revenue <= 0:
        raise ValueError('revenue must be positive')
    return gross_profit / revenue

class Filing:
    def __init__(self, sales: float, cost_of_sales: float) -> None:
        self.sales = sales
        self.cost_of_sales = cost_of_sales

def gross_margin_from_filing(source: Filing) -> float:
    if source.sales <= 0:
        raise ValueError('revenue must be positive')
    return (source.sales - source.cost_of_sales) / source.sales

def label_margin(margin: float) -> str:
    return 'high' if margin >= 0.60 else 'normal'

## Experiment 1 · Replace the implementation

Prediction: will the caller change when the implementation changes but the contract remains stable?

In [ ]:
mapping_margin = gross_margin_from_mapping({'revenue': 100.0, 'gross_profit': 65.0})
filing_margin = gross_margin_from_filing(Filing(100.0, 35.0))
assert mapping_margin == filing_margin == 0.65
assert label_margin(mapping_margin) == label_margin(filing_margin) == 'high'
mapping_margin, filing_margin

## Experiment 2 · Violate the unit invariant

Prediction: what will the unchanged caller do if one implementation returns percent instead of ratio?

In [ ]:
def gross_margin_percent(source: Filing) -> float:
    return 100 * (source.sales - source.cost_of_sales) / source.sales

violating_value = gross_margin_percent(Filing(100.0, 35.0))
print('returned:', violating_value)
print('caller label:', label_margin(violating_value))
print('The type is still float, but the contract is broken.')

## Experiment 3 · Collapse distinct states

Prediction: what information is lost when invalid data and genuine zero margin both become `0.0`?

In [ ]:
def unsafe_margin(source: Mapping[str, float]) -> float:
    revenue = source.get('revenue', 0.0)
    if revenue <= 0:
        return 0.0
    return source.get('gross_profit', 0.0) / revenue

invalid = unsafe_margin({'gross_profit': 10.0})
genuine_zero = unsafe_margin({'revenue': 100.0, 'gross_profit': 0.0})
print({'invalid_data': invalid, 'genuine_zero_margin': genuine_zero})
assert invalid == genuine_zero
print('The abstraction made two task-relevant states indistinguishable.')

## Reflection

For each experiment, record: Purpose, Interface, Invariant, Hidden Detail, prediction, observation, and the smallest contract repair.